# Picto3D — Backend Image → 3D (TripoSR)

Chạy notebook này trên **Google Colab** với GPU:
`Runtime → Change runtime type → Hardware accelerator: T4 GPU`.

Notebook tự động **clone mã nguồn từ GitHub**
[`longtran592005/picto3d`](https://github.com/longtran592005/picto3d) vào
`/content/picto3d` — không cần upload thư mục dự án.

Kết quả cuối cùng là một **public API URL** để dán vào web studio.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Chưa bật GPU. Vào Runtime → Change runtime type → chọn T4 GPU, "
        "bấm Save, sau đó Runtime → Restart session và chạy lại từ đầu."
    )

print("✓ GPU:", torch.cuda.get_device_name(0))
print("  torch", torch.__version__, "| CUDA", torch.version.cuda)

## 1. Lấy mã nguồn từ GitHub

Cell dưới đây sẽ:
- clone repo về `/content/picto3d` nếu chưa có;
- hoặc `git pull` để cập nhật nếu đã clone từ lần chạy trước;
- giữ nguyên thư mục nếu bạn đã upload code thủ công (không có `.git`).

Muốn dùng repo/fork khác, sửa `REPO_URL`. Chạy lại cell này mỗi khi có commit mới.

In [ ]:
%%bash
set -e

REPO_URL="https://github.com/longtran592005/picto3d.git"
PROJECT_DIR="/content/picto3d"

if [ -d "$PROJECT_DIR/.git" ]; then
  echo "↻ Cập nhật mã nguồn trong $PROJECT_DIR"
  git -C "$PROJECT_DIR" pull --ff-only
elif [ -d "$PROJECT_DIR" ]; then
  echo "⚠ $PROJECT_DIR đã tồn tại nhưng không phải git repo — dùng nguyên trạng."
else
  echo "↓ Clone $REPO_URL"
  git clone --depth 1 "$REPO_URL" "$PROJECT_DIR"
fi

ls -la "$PROJECT_DIR"

## 2. Cài đặt dependencies

Nếu thư mục dự án nằm ở chỗ khác, sửa `%cd` bên dưới.

> Không pin `transformers==4.35.0` nữa: bản đó yêu cầu `tokenizers<0.15`,
> không có wheel cho Python mới trên Colab nên pip báo `ResolutionImpossible`.

In [ ]:
%cd /content/picto3d/backend
!ls -la

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
from importlib.util import find_spec

required = [
    "fastapi",
    "uvicorn",
    "trimesh",
    "rembg",
    "scipy",
    "fast_simplification",
    "transformers",
    "huggingface_hub",
]
missing = [name for name in required if find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Thiếu thư viện: {', '.join(missing)} — chạy lại cell pip install phía trên."
    )

import numpy
import transformers

print(
    f"✓ Dependencies OK | numpy {numpy.__version__} | "
    f"transformers {transformers.__version__}"
)

### 2.1 Marching cubes

TripoSR cần `torchmcubes`. Nếu build thất bại (thường gặp trên Colab vì thiếu
`nvcc`), notebook cài `PyMCubes` để backend tự fallback sang CPU — chậm hơn
một chút nhưng vẫn chạy được.

In [ ]:
!pip install -q git+https://github.com/tatsy/torchmcubes.git > /dev/null 2>&1 || (echo '⚠ torchmcubes build thất bại — cài PyMCubes fallback (marching cubes chạy CPU)' && pip install -q PyMCubes)

## 3. Tải mã nguồn TripoSR

`colab_launch.py` tự thêm `/content/TripoSR` vào `PYTHONPATH`.

In [ ]:
%cd /content
!test -d TripoSR || git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git
%cd /content/picto3d/backend

## 4. Chạy server + public tunnel

Cell này sẽ:
1. khởi động FastAPI/Uvicorn ở cổng 8000,
2. tải `cloudflared` và mở quick tunnel,
3. in ra **API URL** dạng `https://xxxx.trycloudflare.com`.

Lần chạy đầu, model TripoSR (~1.5 GB) và model tách nền sẽ được tải về.
Giữ cell này chạy trong lúc dùng web studio.

In [ ]:
!python colab_launch.py --port 8000

## Ghi chú

- **Phải có GPU** (T4). Cell đầu tiên sẽ dừng ngay nếu runtime là CPU.
- **Không đóng tab Colab** khi đang dựng mô hình.
- Đã push commit mới? Dừng cell cuối, chạy lại **cell clone** để `git pull`,
  rồi chạy lại cell server (không cần cài lại dependencies).
- Nếu vừa cài xong thư viện mà import vẫn lỗi: `Runtime → Restart session`
  rồi chạy lại từ cell GPU (bỏ qua bước cài lại nếu đã cài).
- Nếu cloudflared lỗi, chạy `!python colab_launch.py --port 8000 --no-tunnel`
  và dùng một tunnel khác (ngrok: `--tunnel ngrok`, cần `NGROK_AUTHTOKEN`).
- VRAM T4 16 GB đủ cho `mc_resolution=256`, `target_faces=40000`.
- Nếu gặp OOM: giảm chất lượng lưới trong web studio hoặc đặt
  `PICTO3D_CHUNK_SIZE=4096` trước khi chạy cell cuối.
- File `.glb` được lưu trong `backend/outputs/` và tự xoá sau 1 giờ.